
# Frozen Delegations (from First Seen Epoch)

This notebook builds CSV outputs where delegations never change after each delegator's first seen epoch
(typically epoch 0). New delegators appearing later are assigned to their closest DRep at that first epoch
and remain with that DRep thereafter.


In [21]:

import pandas as pd
from pathlib import Path

IN_DIR = Path("csv_out_exp")
OUT_DIR = Path("csv_out_frozen")
OUT_DIR.mkdir(parents=True, exist_ok=True)

dreps = pd.read_csv(IN_DIR / "dreps_state.csv")
deleg = pd.read_csv(IN_DIR / "delegators_state.csv")

assert set(['epoch','drep_id','opinion','stake']).issubset(dreps.columns)
assert set(['epoch','delegator_id','opinion','stake','s']).issubset(deleg.columns)

dreps.head(), deleg.head()

(   epoch drep_id   opinion     stake
 0      0      d1  0.386217  0.488469
 1      0      d2  0.450241  0.377078
 2      0      d3  0.652259  0.627428
 3      0      d4  0.644337  0.641185
 4      0      d5  0.161554  0.544423,
    epoch delegator_id   opinion     stake         s
 0      0           a1  0.802161  0.590105  0.057942
 1      0           a2  0.694992  0.884621  0.656573
 2      0           a3  0.269281  0.027263  0.469737
 3      0           a4  0.708820  0.196704  0.480557
 4      0           a5  0.635183  0.025914  0.906085)

In [22]:

def assign_closest(Ae: pd.DataFrame, De: pd.DataFrame) -> pd.DataFrame:
    a = Ae[['delegator_id','opinion']].rename(columns={'opinion':'op_a'}).copy()
    d = De[['drep_id','opinion']].rename(columns={'opinion':'op_d'}).copy()
    a['key'] = 1; d['key'] = 1
    pairs = a.merge(d, on='key').drop(columns=['key'])
    pairs['distance'] = (pairs['op_a'] - pairs['op_d']).abs()
    nearest = (pairs.sort_values(['delegator_id','distance','drep_id'])
                    .groupby('delegator_id', as_index=False)
                    .first())
    nearest = nearest.rename(columns={'op_d':'drep_opinion'})
    return nearest[['delegator_id','drep_id','drep_opinion','distance']]


In [23]:

# First seen epoch per delegator
first_seen = deleg.groupby('delegator_id', as_index=False)['epoch'].min().rename(columns={'epoch':'first_epoch'})
deleg = deleg.merge(first_seen, on='delegator_id', how='left')

dreps['drep_id'] = dreps['drep_id'].astype(str)
deleg['delegator_id'] = deleg['delegator_id'].astype(str)

epochs = sorted(dreps['epoch'].unique())

frozen_map = {}
delegator_rows = []
drep_rows = []

for e in epochs:
    D = dreps.loc[dreps['epoch'] == e, ['drep_id','opinion','stake']].copy()
    A = deleg.loc[deleg['epoch'] == e, ['delegator_id','opinion','stake','s','first_epoch']].copy()

    # freeze mapping at first seen epoch
    new_ids = [aid for aid, fe in zip(A['delegator_id'], A['first_epoch']) if (aid not in frozen_map) and (fe == e)]
    if new_ids:
        Ae_new = A[A['delegator_id'].isin(new_ids)][['delegator_id','opinion']].copy()
        nearest_new = assign_closest(Ae_new, D)
        for _, row in nearest_new.iterrows():
            frozen_map[row['delegator_id']] = row['drep_id']

    # fallback: if any still unassigned (data quirks), assign now
    missing = [aid for aid in A['delegator_id'] if aid not in frozen_map]
    if missing:
        Ae_new = A[A['delegator_id'].isin(missing)][['delegator_id','opinion']].copy()
        nearest_new = assign_closest(Ae_new, D)
        for _, row in nearest_new.iterrows():
            frozen_map[row['delegator_id']] = row['drep_id']

    # Build per-delegator rows for epoch e
    map_df = pd.DataFrame({'delegator_id': list(A['delegator_id']), 'drep_id': [frozen_map[aid] for aid in A['delegator_id']]}).astype(str)
    D_op = D[['drep_id','opinion']].rename(columns={'opinion':'drep_opinion'})
    joined = A.merge(map_df, on='delegator_id', how='left').merge(D_op, on='drep_id', how='left')
    joined['distance'] = (joined['opinion'] - joined['drep_opinion']).abs()

    for _, r in joined.iterrows():
        delegator_rows.append({
            'epoch': int(e),
            'delegator_id': r['delegator_id'],
            'opinion': float(r['opinion']),
            'stake': float(r['stake']),
            's': float(r['s']),
            'drep_id': r['drep_id'],
            'drep_opinion': float(r['drep_opinion']),
            'distance': float(r['distance']),
        })

    # DRep aggregates
    own = dict(zip(D['drep_id'], D['stake']))
    delegated_stake = joined.groupby('drep_id')['stake'].sum().to_dict()
    indeg = joined.groupby('drep_id')['delegator_id'].count().to_dict()
    avgdist = joined.groupby('drep_id')['distance'].mean().to_dict()

    all_ids = list(D['drep_id'])
    total_Wprime = 0.0
    tmp = []
    for d_id in all_ids:
        del_st = float(delegated_stake.get(d_id, 0.0))
        own_st = float(own.get(d_id, 0.0))
        Wp = own_st + del_st
        total_Wprime += Wp
        tmp.append({
            'epoch': int(e),
            'drep_id': d_id,
            'opinion': float(D.loc[D['drep_id']==d_id, 'opinion'].iloc[0]),
            'stake': own_st,
            'delegated_stake': del_st,
            'indegree': int(indeg.get(d_id, 0)),
            'avg_distance': float(avgdist.get(d_id, 0.0)),
            'Wprime': Wp,
        })
    for row in tmp:
        row['Wprime_share'] = (row['Wprime'] / total_Wprime) if total_Wprime > 0 else 0.0
        drep_rows.append(row)

print("Frozen mappings created for delegators:", len(frozen_map))


Frozen mappings created for delegators: 2000


In [24]:

deleg_frozen = pd.DataFrame(delegator_rows, columns=[
    'epoch','delegator_id','opinion','stake','s','drep_id','drep_opinion','distance'
])
dreps_frozen = pd.DataFrame(drep_rows, columns=[
    'epoch','drep_id','opinion','stake','delegated_stake','indegree','avg_distance','Wprime','Wprime_share'
])

out1 = OUT_DIR / "delegators_state_frozen.csv"
out2 = OUT_DIR / "dreps_state_with_wprime_frozen.csv"
deleg_frozen.to_csv(out1, index=False)
dreps_frozen.to_csv(out2, index=False)

print("Saved:", out1.resolve())
print("Saved:", out2.resolve())


Saved: /Users/Joel/Documents/GitHub/ada_drep/simulation/csv_out_frozen/delegators_state_frozen.csv
Saved: /Users/Joel/Documents/GitHub/ada_drep/simulation/csv_out_frozen/dreps_state_with_wprime_frozen.csv
